# Step 05 — endotypes: clustering patients in module space

An **endotype** is a set of patients sharing a coordinated shift across a set of proteins — a
rectangle in the matrix. Patients are clustered on the standardised eigenproteins of the modules
that carry a clinical association, not on all 7,288 proteins and not on all 25 modules.

**This is the one step with a supervised ingredient, and it is declared rather than hidden.** The
modules were defined without looking at the patients (step 02); restricting to the clinically
associated ones (step 04) uses the traits. Any claim that the endotypes "independently predict"
those traits is therefore circular. What the endotypes can honestly claim is a *partition*, to be
tested against held-out cohorts B and C.

In [ ]:
suppressMessages(library(cluster))
set.seed(42)
e <- readRDS("artifacts/eigengenes_A.rds"); sig <- readRDS("artifacts/sig_modules_A.rds")
S <- scale(e$ME[, paste0("ME", sig), drop = FALSE])

# Silhouette alone will happily reward a partition that isolates one outlier: a
# singleton has silhouette 1 by construction and drags the average up. So score
# each k, but only consider k whose SMALLEST cluster could support a comparison.
MIN_CLUSTER <- 10
fit <- lapply(2:6, function(k) kmeans(S, k, nstart = 50))
tab <- data.frame(
  k       = 2:6,
  sil     = sapply(fit, function(f) mean(silhouette(f$cluster, dist(S))[, 3])),
  smallest= sapply(fit, function(f) min(f$size)))
tab$usable <- tab$smallest >= MIN_CLUSTER
round(tab, 3)

In [ ]:
if (!any(tab$usable)) stop("no k from 2 to 6 gives clusters of at least ", MIN_CLUSTER)
K    <- tab$k[tab$usable][which.max(tab$sil[tab$usable])]
endo <- factor(paste0("E", fit[[K - 1]]$cluster))
c(K = K, silhouette = round(tab$sil[tab$k == K], 3))
table(endo)

## What the silhouette is, and why it needs a floor

Average silhouette width scores how much better each patient fits its own cluster than the nearest
other one. It runs from −1 to 1, and the convention is that below about 0.25 no substantial
structure has been found.

It has one failure mode that matters here: **a cluster of one has silhouette 1 by construction.**
On this data, an unguarded search picked k = 5 with clusters of sizes 5 / 1 / 30 / 4 / 50 — the
score was carried by the singletons, and the "endotypes" were four outliers plus everybody else.
Requiring the smallest cluster to hold at least ten patients removes that, at the cost of possibly
rejecting every k, which is reported rather than worked around.

In [ ]:
w <- readRDS("artifacts/wgcna_A.rds"); mods <- w$mods
Fs <- apply(S, 2, function(x) summary(aov(x ~ endo))[[1]][["F value"]][1])
blocks <- t(sapply(levels(endo), function(g) colMeans(S[endo == g, , drop = FALSE])))
ord <- names(sort(Fs, decreasing = TRUE))
round(blocks[, head(ord, 8)], 2)

## Reading the block means

Read *which* modules separate the endotypes before reading how far apart they are. `blue` holds
1,213 proteins and `black` 150; an eigenprotein over that much of the panel is close to the first
principal component of everything, and a split along it is one dominant axis rather than two
mechanisms.

**The interferon module is not part of the split, and that is the informative result.** It separates
the two endotypes by roughly 0.1 SD — essentially not at all — despite being the module with the
strongest clinical associations in step 04. The interferon axis runs **orthogonal** to whatever the
k-means partition is finding, so the interferon-high patients are a group this partition does not
isolate.

That points somewhere specific: cluster patients on the interferon module alone, rather than on the
full eigenprotein space where 1,213-protein modules dominate the distance.

The cell below locates the module by SOMAmer, not by colour — **WGCNA colours are assigned by module
rank within a single fit and carry no meaning between fits.**

In [ ]:
# Where did the interferon proteins go? Locate the module by the SOMAmer the source
# paper names as its interferon hub, rather than by a colour hardcoded from an earlier run --
# WGCNA colours are assigned by module RANK and mean nothing across fits.
w    <- readRDS("artifacts/wgcna_A.rds")
ifn  <- w$mods[grep("14148", colnames(w$X))[1]]
cat(sprintf("ISG15 seq.14148.2 is in module '%s' (%d proteins)\n", ifn, sum(w$mods == ifn)))

if (paste0("ME", ifn) %in% colnames(blocks)) {
  round(blocks[, paste0("ME", ifn), drop = FALSE], 2)
} else {
  cat(sprintf("'%s' carries no clinical association at FDR 5%%, so it is not in this space.\n", ifn))
}

In [ ]:
saveRDS(list(endo = endo, K = K, scan = tab, F = Fs), "artifacts/endotypes_A.rds")